In [2]:
#imports
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [3]:
#importing the dataset
from pathlib import Path
csv_path = Path("airbnb_newyork.csv")
df = pd.read_csv(csv_path)
df.shape

(48899, 17)

In [4]:
#1 - remove duplicate rows from the dataset
df_2 = df.drop_duplicates()

df_2.shape

(48895, 17)

In [5]:
#2 - identify and delete columns containing only one unique value
count = df_2.nunique()
count

id                                48895
name                              47894
host_id                           37457
host_name                         11451
neighbourhood_group                   5
neighbourhood                       221
latitude                          19048
longitude                         14718
room_type                             3
price                               674
price_level                           2
minimum_nights                      109
number_of_reviews                   394
reviews_per_month                   937
calculated_host_listings_count       47
availability_365                    366
available_online                      1
dtype: int64

In [6]:
col_to_drop = count.loc[count.values == 1].index
col_to_drop

Index(['available_online'], dtype='object')

In [7]:
df_3 = df_2.drop(col_to_drop, axis=1)
df_3

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,price_level,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149.0,low,1.0,9.0,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home,225.0,high,1.0,45.0,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150.0,low,3.0,0.0,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,NaN,89.0,low,1.0,270.0,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home,80.0,low,10.0,9.0,0.10,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48894,36484665,Charming one bedroom - newly renovated rowhouse,8232441,Sabrina,Brooklyn,Bedford-Stuyvesant,40.67853,-73.94995,Private room,70.0,low,2.0,0.0,NaN,2,9
48895,36485057,Affordable room in Bushwick/East Williamsburg,6570630,Marisol,Brooklyn,Bushwick,40.70184,-73.93317,Private room,40.0,low,4.0,0.0,NaN,2,36
48896,36485431,Sunny Studio at Historical Neighborhood,23492952,Ilgar & Aysel,Manhattan,Harlem,40.81475,-73.94867,Entire home,115.0,low,10.0,0.0,NaN,1,27
48897,36485609,43rd St. Time Square-cozy single bed,30985759,Taz,Manhattan,Hell's Kitchen,40.75751,-73.99112,Shared room,55.0,low,1.0,0.0,NaN,6,2


In [8]:
#3 - delete columns
col_to_drop2 = [
    "id", "name", "host_id", "host_name", "neighbourhood_group", "neighbourhood", "latitude", "longitude"]

df_4 = df_3.drop(col_to_drop2, axis=1)
df_4

,room_type,price,price_level,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,Private room,149.0,low,1.0,9.0,0.21,6,365
1,Entire home,225.0,high,1.0,45.0,0.38,2,355
2,Private room,150.0,low,3.0,0.0,NaN,1,365
3,NaN,89.0,low,1.0,270.0,4.64,1,194
4,Entire home,80.0,low,10.0,9.0,0.10,1,0
...,...,...,...,...,...,...,...,...
48894,Private room,70.0,low,2.0,0.0,NaN,2,9
48895,Private room,40.0,low,4.0,0.0,NaN,2,36
48896,Entire home,115.0,low,10.0,0.0,NaN,1,27
48897,Shared room,55.0,low,1.0,0.0,NaN,6,2


In [9]:
#4 - detect missing values and replace
df_4.isnull().sum()

room_type                             6
price                                 5
price_level                           0
minimum_nights                        5
number_of_reviews                    29
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

In [10]:
categorical = df_4.select_dtypes(exclude=[np.number])
numerical = df_4.select_dtypes(include=[np.number])

In [11]:
num = SimpleImputer(strategy="mean")
cat = SimpleImputer(strategy="most_frequent")

In [12]:
categorical_imputed=cat.fit_transform(categorical)
categorical_imputed = pd.DataFrame(categorical_imputed, columns=categorical.columns)
categorical_imputed.isna().sum()

room_type      0
price_level    0
dtype: int64

In [13]:
numerical_imputed = num.fit_transform(numerical)
numerical_imputed = pd.DataFrame(numerical_imputed, columns=numerical.columns)
numerical_imputed.isnull().sum()

price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
dtype: int64

In [14]:
df_5 = pd.concat([numerical_imputed, categorical_imputed], axis=1)
df_5.isnull().sum()

price                             0
minimum_nights                    0
number_of_reviews                 0
reviews_per_month                 0
calculated_host_listings_count    0
availability_365                  0
room_type                         0
price_level                       0
dtype: int64

In [15]:
#5 encoding and type

print("Number of variables requiring encoding:", categorical_imputed.shape[1])

for column in categorical_imputed.columns:
    print(column, ":", categorical_imputed[column].unique())
print("room_type is nominal and price_level is ordinal")

Number of variables requiring encoding: 2
room_type : ['Private room' 'Entire home' 'Shared room']
price_level : ['low' 'high']
room_type is nominal and price_level is ordinal


In [16]:
#6 - utilize ordinal and one-hot encoding
encoder = OrdinalEncoder(categories=[["low", "high"]])
df_5[["price_level"]] = encoder.fit_transform(df_5[["price_level"]])
df_5

,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,room_type,price_level
0,149.0,1.000000,9.0,0.210000,6.0,365.0,Private room,0.0
1,225.0,1.000000,45.0,0.380000,2.0,355.0,Entire home,1.0
2,150.0,3.000000,0.0,1.373221,1.0,365.0,Private room,0.0
3,89.0,1.000000,270.0,4.640000,1.0,194.0,Entire home,0.0
4,80.0,10.000000,9.0,0.100000,1.0,0.0,Entire home,0.0
...,...,...,...,...,...,...,...,...
48890,70.0,2.000000,0.0,1.373221,2.0,9.0,Private room,0.0
48891,40.0,4.000000,0.0,1.373221,2.0,36.0,Private room,0.0
48892,115.0,10.000000,0.0,1.373221,1.0,27.0,Entire home,0.0
48893,55.0,1.000000,0.0,1.373221,6.0,2.0,Shared room,0.0


In [17]:
encoded_room_type = pd.get_dummies(df_5["room_type"])
encoded_room_type = encoded_room_type.astype(int)
encoded_room_type

,Entire home,Private room,Shared room
0,0,1,0
1,1,0,0
2,0,1,0
3,1,0,0
4,1,0,0
...,...,...,...
48890,0,1,0
48891,0,1,0
48892,1,0,0
48893,0,0,1


In [18]:
df_6 = pd.concat([encoded_room_type, df_5], axis=1)
df_7= df_6.drop("room_type", axis=1)
df_7

,Entire home,Private room,Shared room,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,price_level
0,0,1,0,149.0,1.000000,9.0,0.210000,6.0,365.0,0.0
1,1,0,0,225.0,1.000000,45.0,0.380000,2.0,355.0,1.0
2,0,1,0,150.0,3.000000,0.0,1.373221,1.0,365.0,0.0
3,1,0,0,89.0,1.000000,270.0,4.640000,1.0,194.0,0.0
4,1,0,0,80.0,10.000000,9.0,0.100000,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
48890,0,1,0,70.0,2.000000,0.0,1.373221,2.0,9.0,0.0
48891,0,1,0,40.0,4.000000,0.0,1.373221,2.0,36.0,0.0
48892,1,0,0,115.0,10.000000,0.0,1.373221,1.0,27.0,0.0
48893,0,0,1,55.0,1.000000,0.0,1.373221,6.0,2.0,0.0


In [19]:
#7 - z-score standardization

X = df_7.drop("price", axis=1)
y = df_7[["price"]].values

scaler = StandardScaler()

X = scaler.fit_transform(X)

X

array([[-1.04030438,  1.09107283, -0.15588723, ..., -0.03471643,
         1.91625031, -0.50015978],
       [ 0.96125713, -0.9165291 , -0.15588723, ..., -0.15610444,
         1.84027456,  1.9993611 ],
       [-1.04030438,  1.09107283, -0.15588723, ..., -0.18645145,
         1.91625031, -0.50015978],
       ...,
       [ 0.96125713, -0.9165291 , -0.15588723, ..., -0.18645145,
        -0.65173008, -0.50015978],
       [-1.04030438, -0.9165291 ,  6.41489377, ..., -0.03471643,
        -0.84166946, -0.50015978],
       [-1.04030438,  1.09107283, -0.15588723, ..., -0.18645145,
        -0.68212038, -0.50015978]])

In [20]:
#8 - linear regression, 5-fold cross validation and RMSE
lr = LinearRegression()
kf = KFold(n_splits=5)

for train_index, test_index in kf.split(X, y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model = lr.fit(X_train, y_train) 
    pred = model.predict(X_test)
    MSE = mean_squared_error(y_test, pred)
    RMSE = np.sqrt(MSE)
    print("RMSE:", RMSE)
    print("Coefficients:", lr.coef_)



RMSE: 247.28293178681923
Coefficients: [[ 12.69792477 -10.43531381  -7.53083395   6.97515032  -8.22356943
    0.24673825  -9.82175463  13.01534507  98.83708722]]
RMSE: 186.3645950125383
Coefficients: [[ 12.66091172 -10.42702914  -7.43644179   3.9167693   -5.20132725
   -1.03678364  -9.41288721  11.92321076 102.11817836]]
RMSE: 184.67546133782466
Coefficients: [[ 12.60100235 -10.48138511  -7.0618653    6.25640649  -4.68632523
   -0.71833499  -9.46624271  10.36383312 100.61350909]]
RMSE: 199.82383312252117
Coefficients: [[ 12.2473722  -10.11235338  -7.10878099   5.1248493   -5.40911746
   -0.35424147  -8.71325926  11.99646977  99.59802285]]
RMSE: 236.50117598566942
Coefficients: [[ 12.52155029 -10.66177251  -6.21063891   2.53527727  -6.32448613
    0.95088855 -12.9359516   10.72015407  98.39778109]]


In [21]:
#9 - Ridge regression, 5-fold cross validation and RMSE
rr = Ridge(alpha=1)
kf = KFold(n_splits=5)

for train_index, test_index in kf.split(X, y):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train the Ridge model
    model = rr.fit(X_train, y_train)
    # Predict
    pred = model.predict(X_test)

    # Calculate RMSE
    MSE = mean_squared_error(y_test, pred)
    RMSE = np.sqrt(MSE)

    print("RMSE:", RMSE)
    print("Coefficients:", rr.coef_)

RMSE: 247.28294783370904
Coefficients: [ 12.69831179 -10.43570658  -7.53081663   6.97488938  -8.22331171
   0.24657838  -9.82105748  13.01525076  98.83414273]
RMSE: 186.36444403056885
Coefficients: [ 12.66131431 -10.42743392  -7.43643697   3.91659143  -5.20137335
  -1.03679621  -9.41223937  11.92309874 102.11519618]
RMSE: 184.67542116751832
Coefficients: [ 12.60139216 -10.48177841  -7.06186077   6.256195    -4.68638874
  -0.71833194  -9.46560742  10.36378204 100.61060384]
RMSE: 199.82385880147507
Coefficients: [ 12.2477421  -10.11273736  -7.10874141   5.12467864  -5.40917094
  -0.35421512  -8.71266323  11.99638082  99.59512964]
RMSE: 236.50117027837666
Coefficients: [ 12.5218909  -10.66212852  -6.21059276   2.53512499  -6.32427361
   0.95064574 -12.93417634  10.71987282  98.39472973]


In [23]:
print("Intercept:", rr.intercept_)

Intercept: [152.57781706]


In [22]:
#ridge regression reduces overfitting, especially when predictors are highly correlated, there are too many predictors or there is an ill-conditioned X, compared to linear regression.
#linear regression estimates coefficients by minimizing the sum squared errors, where ridge regression adds a penalty based on the squared coefficients that shrinks them towards zero and can reduce overfitting.
#the lowest RMSE is the third fold with an RMSE of 184.67542116751832
#the coefficients associated witht the lowest RMSE i are: [ 12.60139216(Entire home) -10.48177841(Private Room) -7.06186077(Shared room)  6.256195(minimum nights) -4.68638874(number of reviews) -0.71833194(reviews per month)
 #-9.46560742(calculated host listings count) 10.36378204(availability 365)   100.61060384(price_level)]

#linear regression formula: Prediction Price = 152.578 + 12.60139216(zEntire home) - 10.48177841(zPrivate Room) - 7.06186077(zShared room) + 6.256195(zMinimum nights) - 4.68638874(zNumber of reviews) - 0.71833194(zReviews per month)
# - 9.46560742(zCalculated host listings count) + 10.36378204(zAvailability 365) + 100.61060384(zPrice-level)

#the x values are the standardized version of the predictor, whilst the predicted y (price) remains in original units